In [0]:
# IMPORTANT:  
#
#   The methods to read files were modified to read/write in volume instead of s3 for DEV and QA. 
#   For example, if you go and execute a job read in DEV environment from:
#                         's3://memberanalytics-data-out-prod/'            +  'ASSIGNMENTS/cdsa/assn_output/PROD/MMPC19FY26_PROD/final_2025-11-11/mail_subset`
#   it will go look at:   '/Volumes/datascience_ea_dev/pe/outputs_for_s3/' +  'ASSIGNMENTS/cdsa/assn_output/PROD/MMPC19FY26_PROD/final_2025-11-11/mail_subset'`

#   The files in volume should be copied from s3 before running the process. 


# LL Note:  VERSION_MAP is currently not used given the yml config, check if this need to be migrated to dbx source
#           `VERSION_MAP: 's3://memberanalytics-data-out-prod/ASSIGNMENTS/ campaigns/FY21/BBM12FY21/Coupon_Input/version map bbm12.csv'`

In [0]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ../../config/utils

In [0]:
# Legacy note previous to LL migration:     
#   TODO: [] Convert trial calculation to member DNA 

import sys
sys.path.append("..")
sys.path.append("../..")

import pyspark.sql.functions as sqlf
from lib_assignment.assn_utils import (
    check_cpn_nbr_or_version,
    read_subset_and_cast,
    subset_by_time,
    env_path,
)
import lib_assignment.assn_io as assn_io
from lib_assignment.checks import check_execution_overwrite

In [0]:

# Params 'output_vol' and 'environment' are configured in variables when this is exec: %run ../../config/utils
job = assn_io.JobManager("MailFile", "../config/config_template.yml", spark, output_vol, environment)


if job.config.params["run_type"].lower() == "prod":

    # avoid overwritting an existing process output
    check_execution_overwrite( job,  paths_to_check=[job.config.paths["MAILFILE"]] )


# Replace those inputs that can be pulled from the Databricks migrated sources:
job.config.paths['RAW_MEMBER']  = silver_master_member_extended
job.config.paths['CUBE']        = fs_customer_cube_full

In [0]:
# while some sources are migrated into databricks we need for the pull to work seamlessly for volumes in dev/qa and s3 in prod
# this way we encapsulate the edit here
def read_subset_and_cast__temp(
    file_path, file_type, subset_cols=None, time_part=None, time_part_val=None
):
    df = read_subset_and_cast( env_path(file_path, output_vol, environment) , file_type, subset_cols, time_part, time_part_val)

    return df

In [0]:
print("1. Reading input data...")

data = read_subset_and_cast__temp( job.config.paths["MAIL_POPULATION_ASSIGNMENT"], "csv" )

renames = data.columns

for name in renames:
    data = data.withColumnRenamed(name, name.upper())

cols = ["MBRSHP_NBR", "MBRSHP_SID"]

mail_list = read_subset_and_cast__temp( job.config.paths["MAIL_LIST"], "csv")  

if "MBRSHP_SID" in mail_list.columns:
    mbr_lkup = mail_list.select(*cols)
else:
    mbr_lkup = read_subset_and_cast( job.config.paths['RAW_MEMBER'], "table", cols   )  


if "MBRSHP_NBR" not in [x.upper() for x in data.columns]:
    data = data.join(mbr_lkup, "MBRSHP_SID", "left")

cols = [
    "MBRSHP_SID",
    "LAST_FIFTY-TWO_WEEK_TRIPS",
    "LFIFTY-TWOW_SPEND_IN_STORE",
    "FISCAL_WEEK_END",
]

cube = read_subset_and_cast( job.config.paths["CUBE"], "table", cols) # update 'table' to 'parquet'  if we are reading from s3

cube = subset_by_time(  cube, job.config.params["assignment_date"], "fiscal_week"  )

cube = cube.withColumn(
    "LAST_FIFTY-TWO_WEEK_TRIPS",
    sqlf.when(sqlf.col("LAST_FIFTY-TWO_WEEK_TRIPS").isNull(), 0).otherwise(
        sqlf.col("LAST_FIFTY-TWO_WEEK_TRIPS")
    ),
)

cube = cube.withColumn(
    "LFIFTY-TWOW_SPEND_IN_STORE",
    sqlf.when(
        sqlf.col("LFIFTY-TWOW_SPEND_IN_STORE").isNull(), 0
    ).otherwise(sqlf.col("LFIFTY-TWOW_SPEND_IN_STORE")),
)

In [0]:
if "bbm" in job.config.params["campaign"].lower():

    if job.config.paths.get("VERSION_MAP") is not None:
        
        job.data.read("version_map", "VERSION_MAP", filetype="csv")  # LL Note:  currently not used, check if this need to be migrated to dbx source

        version_map = job.data.tables["version_map"]

        # if the cpn_nbr assigned is a letter - it is actually the version
        version_map = version_map.withColumnRenamed("CPN1", "CPN_NBR")
        data = data.join(version_map, "CPN_NBR", "left")
        data = data.withColumn(
            "VERSION",
            sqlf.when(
                sqlf.col("VERSION").isNull(), sqlf.col("CPN_NBR")
            ).otherwise(sqlf.col("VERSION")),
        )
    else:
        data = data.withColumn("VERSION", sqlf.col("CPN_NBR"))
    data = check_cpn_nbr_or_version(data, "CPN_NBR")
else:
    data = data.withColumn("VERSION", sqlf.lit(""))



if "DECILE" not in data.columns:
    cols = ["MBRSHP_NBR", "decile"]
    job.data.read("decile", "MAIL_LIST", filetype="csv", cols=cols)  
    dec = job.data.tables["decile"]
    data = data.join(dec, "MBRSHP_NBR", "left")
    data = data.withColumnRenamed("decile", "DECILE")

In [0]:
print("2. Generate mailfile dataset...")
#----------------------------------------------------------------------------------------------------

mail = data.withColumn("slot", sqlf.concat(sqlf.lit("CPN"), data.SLOT_NBR))

if "mail_flag" not in [x.lower() for x in mail.columns]:
    mail = mail.withColumn("mail_flag", sqlf.lit("1"))

mail = mail.withColumn(
    "mail_flag",
    sqlf.when(sqlf.col("mail_flag") != 0, "CIRC").otherwise("NO MAIL"),
)

print("Not Mailed members:  {}".format( mail.filter("mail_flag == 'NO MAIL'").count()))
print("Mailed members:      {}".format(mail.filter("mail_flag == 'CIRC'").count()))

mail = (
    mail.groupBy("MBRSHP_NBR", "MBRSHP_SID", "CELL_ID", "mail_flag")
    .pivot("slot")
    .agg(sqlf.first("CPN_NBR"))
)

# Combine with cube and data 
mail = mail.join(cube, "MBRSHP_SID", "left")
mail = mail.join(
    data.select("MBRSHP_SID", "VERSION", "DECILE").distinct(),
    "MBRSHP_SID",
    "left",
)
mail = mail.repartition(1) # to be saved as single file?


# Build the column list:
#----------------------------------------------------------------------------------------------------
data = data.withColumn("SLOT_NBR", data.SLOT_NBR.cast("integer"))
slot_list = sorted(
    data.select("SLOT_NBR").distinct().toPandas()["SLOT_NBR"]
)
slot_name = ["CPN" + str(s) for s in slot_list]

col_list = (
    ["MBRSHP_NBR", "CELL_ID", "mail_flag", "VERSION"]
    + slot_name
    + [
        "MBRSHP_SID",
        "DECILE",
        "LAST_FIFTY-TWO_WEEK_TRIPS",
        "LFIFTY-TWOW_SPEND_IN_STORE",
    ]
)


# Select only the columns we need
#----------------------------------------------------------------------------------------------------
mail = mail.select(col_list)

In [0]:
print("3. Writing output...")
print("writing mailfile...")

job.data.add("final_mailhouse", mail) # here we add the df to the job object


In [0]:
# LL note here:  while working in dev and qa this will write to volume instead of s3:
try:
    job.data.write( # write the df to the csv
        "final_mailhouse",
        "MAILFILE", #`"MAILFILE" s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/assn_output/PROD/MMPC19FY26_PROD/final_2025-11-11/final_mailhouse`
        mode="overwrite",
        singlefile=True,
        ftype="csv",
        dbutils=dbutils
    )

except Exception as e:
    print(f"Error writing to s3: {type(e).__name__}: {e}")